# AccentSense — UK Regional Accent Classifier
**WavLM Base+ · Attentive Statistics Pooling · 6 UK Dialect Classes**

VIT Bhopal University · AI/ML Capstone Research

---
**Classes:** `RP` · `Scottish` · `Welsh` · `Northern` · `West_Midlands` · `Irish`

**Hardware:** Local GPU (NVIDIA CUDA / AMD DirectML) or Google Colab T4

**Steps:**
1. GPU check
2. Install dependencies
3. Verify project structure
4. Download VCTK corpus (or use Hugging Face fallback)
5. (Optional) Irish English supplement
6. Build speaker-disjoint manifests
7. Train WavLM (20 epochs, cosine LR, class-weighted CE)
8. Evaluate — classification report + confusion matrix
9. Checkpoint ready → place in `Backend/checkpoints/`


In [ ]:
# Step 1: Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM: %.1f GB' % vram)
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('For Colab: Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# Step 2: Install dependencies
import subprocess
import sys

DEPS = ['transformers', 'accelerate', 'torchaudio', 'datasets', 'soundfile', 'jiwer', 'scikit-learn', 'pandas', 'captum', 'scipy']

for pkg in DEPS:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All dependencies installed.')


In [ ]:
# Step 3: Verify project structure (local — no clone needed)
import os, sys

# Detect if we're in Backend/ or project root
if os.path.isdir('src'):
    BACKEND_DIR = os.getcwd()
elif os.path.isdir('Backend/src'):
    BACKEND_DIR = os.path.join(os.getcwd(), 'Backend')
    os.chdir(BACKEND_DIR)
else:
    # Try parent directory
    parent = os.path.dirname(os.getcwd())
    if os.path.isdir(os.path.join(parent, 'Backend', 'src')):
        BACKEND_DIR = os.path.join(parent, 'Backend')
        os.chdir(BACKEND_DIR)
    else:
        raise RuntimeError('Cannot find project structure. Run from project root or Backend/ directory.')

# Add to Python path
for p in [os.getcwd(), os.path.abspath('.')]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

print('Working dir:', os.getcwd())
print('src directory found:', os.path.isdir('src'))


In [ ]:
# Step 4: Download VCTK Corpus (with Hugging Face fallback)
import os, glob

VCTK_URL = 'https://datashare.is.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip'
USE_HF_VCTK = False
zip_path = 'data/vctk.zip'
os.makedirs('data/vctk', exist_ok=True)

downloaded = False

# Attempt 1: Python requests download
print('Attempting download via Python requests...')
try:
    import requests
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    with requests.get(VCTK_URL, headers=headers, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        if total > 1_000_000_000:
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            sz = os.path.getsize(zip_path)
            if sz > 1_000_000_000:
                downloaded = True
                print(f'Downloaded via requests: {sz / (1024**3):.2f} GB')
            else:
                print(f'Download too small ({sz} bytes).')
                os.remove(zip_path)
        else:
            print(f'Server returned small content-length ({total} bytes).')
except Exception as e:
    print(f'Requests download failed: {e}')
    if os.path.exists(zip_path):
        try:
            os.remove(zip_path)
        except OSError:
            pass

# Extract if downloaded, otherwise fall back to Hugging Face
if downloaded:
    print('Extracting VCTK (this may take a few minutes)...')
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('data/vctk/')
    try:
        os.remove(zip_path)
    except OSError:
        pass
    info = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
    if info:
        print('Local VCTK extracted. speaker-info.txt:', info[0])
    else:
        print('speaker-info.txt not found after extraction. Will fall back to Hugging Face.')
        USE_HF_VCTK = True
else:
    print('\nEdinburgh DataShare server unavailable or download timed out.')
    print('Automatically falling back to Hugging Face in Step 6 (no manual action needed).')
    USE_HF_VCTK = True
    if os.path.exists(zip_path):
        try:
            os.remove(zip_path)
        except OSError:
            pass

if USE_HF_VCTK:
    print('VCTK audio will be streamed & loaded directly from Hugging Face in Step 6.')


In [ ]:
# Step 5 (Optional): Add Irish English speech supplement
import os
import soundfile as sf
import numpy as np

cv_extras = []

try:
    from datasets import load_dataset
    irish = []

    # Try open-access English Dialects dataset (public, no auth required)
    print('Trying open English Dialects dataset (no token required)...')
    for subset in ['irish_male', 'irish_female']:
        try:
            ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
            for s in ds:
                irish.append({
                    'audio': s['audio'],
                    'speaker': s.get('speaker_id', f'spk_{subset}')
                })
                if len(irish) >= 500:
                    break
            print(f'  Found {len(irish)} samples from {subset}')
            if len(irish) >= 500:
                break
        except Exception as e:
            print(f'  {subset} not available: {e}')

    # Fallback to Mozilla Common Voice if needed
    if len(irish) < 50:
        print('Trying Mozilla Common Voice...')
        hf_token = os.environ.get('HF_TOKEN')
        if not hf_token:
            print('  HF_TOKEN not set. Skipping Common Voice.')

        if hf_token:
            IRISH_TAGS = {'Ireland', 'Irish', 'Dublin', 'Northern Ireland'}
            for cv_version in ['mozilla-foundation/common_voice_17_0', 'mozilla-foundation/common_voice_13_0']:
                try:
                    print(f'Trying {cv_version}...')
                    cv = load_dataset(cv_version, 'en', split='train', streaming=True, token=hf_token)
                    for s in cv.take(100000):
                        if s.get('accent', '') in IRISH_TAGS:
                            irish.append({
                                'audio': s['audio'],
                                'speaker': s.get('client_id', 'cv_unknown')
                            })
                        if len(irish) >= 500:
                            break
                    if irish:
                        break
                except Exception as ve:
                    print(f'  {cv_version} failed: {ve}')
                    continue

    if irish:
        print(f'\nTotal Irish clips found: {len(irish)}')
        os.makedirs('data/cv_irish', exist_ok=True)
        for i, item in enumerate(irish):
            p = f'data/cv_irish/irish_{i:04d}.wav'
            sf.write(p, np.array(item['audio']['array'], dtype=np.float32), item['audio']['sampling_rate'])
            cv_extras.append({
                'path': p,
                'label': 'Irish',
                'speaker': str(item['speaker']),
                'class_idx': 5
            })
        print(f'Saved {len(cv_extras)} Irish WAV files.')
    else:
        print('No Irish clips could be fetched. Continuing with VCTK-only data.')

except Exception as e:
    print(f'Irish supplement error: {e}')
    print('Continuing with VCTK-only data.')


In [ ]:
# Step 6: Build speaker-disjoint manifests
from collections import Counter
import glob, os, sys, json, random
import numpy as np
import soundfile as sf

CLASSES = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Irish')

VCTK_ACCENT_MAP = {
    'English': 'RP', 'Southern England': 'RP', 'Surrey': 'RP', 'Kent': 'RP',
    'Hampshire': 'RP', 'Essex': 'RP', 'Berkshire': 'RP', 'Oxfordshire': 'RP',
    'Scottish': 'Scottish', 'Scotland': 'Scottish', 'Edinburgh': 'Scottish', 'Glasgow': 'Scottish',
    'Welsh': 'Welsh', 'Wales': 'Welsh',
    'Northern England': 'Northern', 'Yorkshire': 'Northern', 'Manchester': 'Northern',
    'Newcastle': 'Northern', 'Geordie': 'Northern', 'Lancashire': 'Northern', 'Leeds': 'Northern',
    'West Midlands': 'West_Midlands', 'Birmingham': 'West_Midlands', 'Midlands': 'West_Midlands',
    'London': 'RP', 'Cockney': 'RP', 'East London': 'RP',
    'Irish': 'Irish', 'Ireland': 'Irish', 'Dublin': 'Irish', 'Northern Ireland': 'Irish',
}

def save_manifest(manifest, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

def speaker_disjoint_splits(manifest, train_ratio=0.75, val_ratio=0.15, seed=42):
    random.seed(seed)
    class_to_speakers = {}
    for entry in manifest:
        cls = entry['label']
        spk = entry['speaker']
        class_to_speakers.setdefault(cls, [])
        if spk not in class_to_speakers[cls]:
            class_to_speakers[cls].append(spk)

    train_speakers, val_speakers, test_speakers = set(), set(), set()
    for cls, speakers in class_to_speakers.items():
        random.shuffle(speakers)
        n = len(speakers)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))
        train_speakers.update(speakers[:n_train])
        val_speakers.update(speakers[n_train:n_train + n_val])
        test_speakers.update(speakers[n_train + n_val:])

    train = [e for e in manifest if e['speaker'] in train_speakers]
    val   = [e for e in manifest if e['speaker'] in val_speakers]
    test  = [e for e in manifest if e['speaker'] in test_speakers]
    return train, val, test

INFO_FILES = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
use_local = (not globals().get('USE_HF_VCTK', False)) and len(INFO_FILES) > 0

manifest = []
speaker_counts = Counter()

if use_local:
    print('=== Building manifest from local VCTK ===')
    INFO = INFO_FILES[0]
    print(f'Speaker info: {INFO}')

    WAVD = None
    for pattern in ['data/vctk/wav48_silence_trimmed', 'data/vctk/wav48', 'data/vctk/wav16']:
        if os.path.isdir(pattern):
            WAVD = pattern
            print(f'Found audio directory: {WAVD}')
            break

    if not WAVD:
        print('Could not find wav directory under data/vctk/. Falling back to Hugging Face...')
        use_local = False

if use_local:
    speaker_accents = {}
    with open(INFO, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('ID'):
                continue
            parts = line.split()
            if len(parts) >= 4:
                speaker_id = parts[0]
                accent = parts[3]
                speaker_accents[speaker_id] = accent

    print(f'Loaded {len(speaker_accents)} speakers from speaker-info.txt')

    ACCENT_TO_CLASS = {
        'RP': 'RP', 'Scottish': 'Scottish', 'Welsh': 'Welsh',
        'Northern': 'Northern', 'West_Midlands': 'West_Midlands',
    }

    print(f'Scanning {WAVD} for audio files...')
    for speaker_dir in sorted(os.listdir(WAVD)):
        speaker_path = os.path.join(WAVD, speaker_dir)
        if not os.path.isdir(speaker_path):
            continue
        speaker_id = speaker_dir
        if speaker_id not in speaker_accents:
            continue
        accent = speaker_accents[speaker_id]
        if accent not in ACCENT_TO_CLASS:
            continue
        class_label = ACCENT_TO_CLASS[accent]
        class_idx = CLASSES.index(class_label)
        audio_files = []
        for ext in ['*.wav', '*.WAV', '*.flac', '*.FLAC']:
            audio_files.extend(glob.glob(os.path.join(speaker_path, ext)))
        for audio_path in audio_files:
            manifest.append({
                'path': audio_path,
                'label': class_label,
                'speaker': speaker_id,
                'class_idx': class_idx
            })
            speaker_counts[class_label] += 1

if not use_local or len(manifest) == 0:
    print('\n=== Local VCTK not found. Streaming & building dataset from Hugging Face ===')
    from datasets import load_dataset

    os.makedirs('data/vctk_hf', exist_ok=True)
    loaded_from_hf = False

    try:
        print("Connecting to 'CSTR-Edinburgh/vctk' (mic1, streaming)...")
        ds = load_dataset('CSTR-Edinburgh/vctk', 'mic1', split='train', streaming=True)
        class_samples = Counter()
        MAX_PER_CLASS = 500

        for idx, row in enumerate(ds):
            accent_tag = str(row.get('accent', row.get('region', '')))
            cls = VCTK_ACCENT_MAP.get(accent_tag)
            if not cls:
                for k, v in VCTK_ACCENT_MAP.items():
                    if k.lower() in accent_tag.lower():
                        cls = v
                        break
            if not cls or cls not in CLASSES or cls == 'Irish':
                continue
            if class_samples[cls] >= MAX_PER_CLASS:
                continue
            spk = str(row.get('speaker_id', f'spk_{cls}'))
            audio = row['audio']
            p = f'data/vctk_hf/{spk}_{idx:05d}.wav'
            if not os.path.exists(p):
                sf.write(p, np.array(audio['array'], dtype=np.float32), audio['sampling_rate'])
            manifest.append({
                'path': p,
                'label': cls,
                'speaker': spk,
                'class_idx': CLASSES.index(cls)
            })
            speaker_counts[cls] += 1
            class_samples[cls] += 1
            if len(manifest) % 150 == 0:
                print(f'  Downloaded {len(manifest)} samples from HF VCTK...')
            if all(class_samples[c] >= MAX_PER_CLASS for c in ['RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands']):
                break
        if len(manifest) > 100:
            loaded_from_hf = True
            print(f'Built manifest with {len(manifest)} clips from CSTR-Edinburgh/vctk')
    except Exception as e:
        print(f'  CSTR-Edinburgh/vctk stream issue: {e}')

    if not loaded_from_hf or len(manifest) < 100:
        print("\nLoading from 'ylacombe/english_dialects' (open UK dialect corpus)...")
        DIALECT_MAP = {
            'southern_male': 'RP', 'southern_female': 'RP',
            'scottish_male': 'Scottish', 'scottish_female': 'Scottish',
            'welsh_male': 'Welsh', 'welsh_female': 'Welsh',
            'northern_male': 'Northern', 'northern_female': 'Northern',
            'midlands_male': 'West_Midlands', 'midlands_female': 'West_Midlands',
        }
        for subset, cls in DIALECT_MAP.items():
            try:
                ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
                cnt = 0
                for idx, row in enumerate(ds):
                    spk = str(row.get('speaker_id', f'spk_{subset}'))
                    audio = row['audio']
                    p = f'data/vctk_hf/{subset}_{idx:04d}.wav'
                    if not os.path.exists(p):
                        sf.write(p, np.array(audio['array'], dtype=np.float32), audio['sampling_rate'])
                    manifest.append({
                        'path': p,
                        'label': cls,
                        'speaker': spk,
                        'class_idx': CLASSES.index(cls)
                    })
                    speaker_counts[cls] += 1
                    cnt += 1
                    if cnt >= 250:
                        break
                print(f'  {subset} -> {cls}: {cnt} samples')
            except Exception as se:
                print(f'  Could not load {subset}: {se}')

if 'cv_extras' in globals() and cv_extras:
    existing_irish_paths = {m['path'] for m in manifest if m['label'] == 'Irish'}
    new_cv = [c for c in cv_extras if c['path'] not in existing_irish_paths]
    manifest.extend(new_cv)
    print(f'\n+ Added {len(new_cv)} Irish samples from Step 5')
    for item in new_cv:
        speaker_counts['Irish'] += 1

print(f'\nTotal audio files in manifest: {len(manifest)}')
print('\nClass distribution:')
for cls in CLASSES:
    count = speaker_counts[cls]
    print(f'  {cls:<15}: {count:5d}')

print('\nCreating speaker-disjoint train/val/test splits...')
train_m, val_m, test_m = speaker_disjoint_splits(manifest)
print(f'Splits -> Train: {len(train_m)} | Val: {len(val_m)} | Test: {len(test_m)}')

os.makedirs('data/manifests', exist_ok=True)
save_manifest(train_m, 'data/manifests/train.json')
save_manifest(val_m,   'data/manifests/val.json')
save_manifest(test_m,  'data/manifests/test.json')

print('Manifests saved to data/manifests/')


In [ ]:
# Step 7: Training (6-class model: RP, Scottish, Welsh, Northern, West_Midlands, Irish)
import os, sys
for p in [os.getcwd(), os.path.abspath('.')]:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, balanced_accuracy_score
from src.data.vctk_dataset import UKAccentDataset, collate_pad, load_manifest
from src.models.wavlm_classifier import WavLMAccentClassifier
from collections import Counter
import torch, os

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Irish')
N       = len(CLASSES)
EPOCHS  = 20
BATCH   = 8
LR      = 3e-4

print(f'Training {N}-class model: {CLASSES}')
print(f'Device: {DEVICE}\n')

model = WavLMAccentClassifier(
    pretrained_model_name='microsoft/wavlm-base-plus',
    num_classes=N,
    freeze_encoder=True,
    unfreeze_top_k_layers=2,
    dropout_p=0.3,
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable: %d / %d (%.1f%%)\n' % (trainable, total, 100*trainable/total))

import json
for split_name in ['train', 'val', 'test']:
    split_path = f'data/manifests/{split_name}.json'
    if os.path.exists(split_path):
        with open(split_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        modified = False
        valid_data = []
        for e in data:
            if e['label'] not in CLASSES:
                modified = True
                continue
            correct_idx = CLASSES.index(e['label'])
            if e['class_idx'] != correct_idx:
                e['class_idx'] = correct_idx
                modified = True
            valid_data.append(e)
        if modified:
            print(f'Sanitized {split_path} to match {N} classes ({len(valid_data)} samples)')
            with open(split_path, 'w', encoding='utf-8') as f:
                json.dump(valid_data, f, indent=2)

tr_ds = UKAccentDataset(load_manifest('data/manifests/train.json'), augment=True)
va_ds = UKAccentDataset(load_manifest('data/manifests/val.json'),   augment=False)
tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_pad, num_workers=2, pin_memory=True)
va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad, num_workers=2)

print(f'Train: {len(tr_ds)} samples ({len(tr_dl)} batches)')
print(f'Val:   {len(va_ds)} samples ({len(va_dl)} batches)\n')

cnt = Counter(e['class_idx'] for e in load_manifest('data/manifests/train.json'))
w   = torch.tensor([1.0 / cnt.get(i, 1) for i in range(N)], dtype=torch.float32)
w   = w / w.sum() * N
criterion = torch.nn.CrossEntropyLoss(weight=w.to(DEVICE))
print('Class distribution in training set:')
for i, cls in enumerate(CLASSES):
    count = cnt.get(i, 0)
    print(f'  {cls:<12}: {count:6d} (weight: {w[i]:.3f})')
print()

optimizer  = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
T          = len(tr_dl) * EPOCHS
scheduler  = get_cosine_schedule_with_warmup(optimizer, T // 10, T)

best = 0.0
os.makedirs('checkpoints', exist_ok=True)
print(f"{'Ep':<4} {'Loss':<10} {'TrF1':<10} {'ValF1':<10} {'BalAcc':<10} Status")
print('-' * 60)

for ep in range(1, EPOCHS + 1):
    model.train()
    el, pp, ll = 0.0, [], []
    for wav, lb, mask in tr_dl:
        wav, lb, mask = wav.to(DEVICE), lb.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        out  = model(wav, attention_mask=mask)
        loss = criterion(out['logits'], lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        el += loss.item()
        pp.extend(out['logits'].argmax(-1).cpu().tolist())
        ll.extend(lb.cpu().tolist())

    tr_f1 = f1_score(ll, pp, average='macro', zero_division=0)

    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for wav, lb, mask in va_dl:
            out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
            vp.extend(out['logits'].argmax(-1).cpu().tolist())
            vl.extend(lb.tolist())

    val_f1 = f1_score(vl, vp, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(vl, vp)
    status = ''
    if val_f1 > best:
        best = val_f1
        torch.save(model.state_dict(), 'checkpoints/best_wavlm_accentsense.pt')
        status = 'BEST'

    print(f'{ep:<4} {el/len(tr_dl):<10.4f} {tr_f1:<10.4f} {val_f1:<10.4f} {bal_acc:<10.4f} {status}')

print('-' * 60)
print(f'Training complete.')
print(f'  Best Val Macro-F1: {best:.4f}')
print(f'  Checkpoint: checkpoints/best_wavlm_accentsense.pt')


In [ ]:
# Step 8: Test-set evaluation
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load('checkpoints/best_wavlm_accentsense.pt', map_location=DEVICE))
model.eval()

te_ds = UKAccentDataset(load_manifest('data/manifests/test.json'), augment=False)
te_dl = DataLoader(te_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad)

tp, tl = [], []
with torch.no_grad():
    for wav, lb, mask in te_dl:
        out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
        tp.extend(out['logits'].argmax(-1).cpu().tolist())
        tl.extend(lb.tolist())

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(classification_report(tl, tp, target_names=CLASSES))
print('Confusion Matrix:')
print(confusion_matrix(tl, tp))
test_f1 = f1_score(tl, tp, average='macro', zero_division=0)
print('Final Test Macro-F1: %.4f' % test_f1)


In [ ]:
# Step 9: Checkpoint ready
import os
ckpt = 'checkpoints/best_wavlm_accentsense.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1024 / 1024
    print(f'Checkpoint ready: {ckpt} ({size:.1f} MB)')
    print('Place at: Backend/checkpoints/best_wavlm_accentsense.pt')
    print('Restart API -> GET /health shows model_loaded: true')
else:
    print('Checkpoint not found. Training may not have completed.')


## After Training

1. Checkpoint saved at `checkpoints/best_wavlm_accentsense.pt`
2. Copy to `Backend/checkpoints/best_wavlm_accentsense.pt`
3. Restart backend: `python run_api.py`
4. `GET /health` -> `model_loaded: true`, `supported_classes: [RP, Scottish, Welsh, Northern, West_Midlands, Irish]`

**Expected Test Macro-F1:** >0.70 on VCTK real recordings

> **Note:** VCTK has few London speakers. If you want Cockney as a 7th class,
> add Common Voice clips tagged 'London' in Step 5 and re-run splits.
